I wanted to put my data into a dataframe so I could work with it easier, clean it, and get it ready for machine learning. Pandas is fantastic for EDA, which made is the main reason why I use it. I read the CSV in as a Dataframe, I then printed the first several rows to ensure it was read in properly.

In [1]:
import pandas as pd

df = pd.read_csv('C:\\Users\\hessk\\OneDrive\\Desktop\\DSC630\\residential_solar_installations_panel_data_420counties_test.csv',
                 dtype=str, low_memory=False)

print(df)

      blockgroup_FIPS cumulative num of residential PVs by 2010  \
0         10090501011                                         0   
1         10090501012                                         0   
2         10090501013                                         0   
3         10090501014                                         0   
4         10090501015                                         0   
...               ...                                       ...   
54488      5.6025E+11                                         0   
54489      5.6025E+11                                         0   
54490      5.6025E+11                                         0   
54491      5.6025E+11                                         0   
54492      5.6025E+11                                         0   

      cumulative num of residential PVs by 2011  \
0                                             0   
1                                             0   
2                                         

I wanted to set my columns so if I needed to change any of them I could change it and re-run my code.

In [41]:
SOURCE_COL = "cumulative_num_of_residential_pvs_by_2017"
TARGET_COL = globals().get("TARGET_COL", "total_installs_detected_years")
SEED = globals().get("SEED", 42)
QUICK = globals().get("QUICK", True)
N_ITER = globals().get("N_ITER", 20)
SENTINELS = globals().get("SENTINELS", [-2147483648])


argparse - allows me to set/define flags and add arguments
os - allows me to work with different files on my computer
re - allows me to work with groups, and find/search the data easier
pathlib - easy use for iterations, and makes chaining different operations easier
matplotlib - I imported for all the plotting capabilities that it has. Simple visualization library
numpy - makes working with numbers easier and faster
seaborn - deeper level of visualizations, with more complex options.

This starts my cleanup of the dataframe. I start by normalizing the colums to ensure they are all in the same type of format. All lowercase, removing symbols, and making sure the column names are unique. The reason it is so important to do this is because when pulling a column in, it makes things so much easier if they are all similar in format. Not having to worry about leading or trailing spaces, not having to worry about capitalizations or symbols. 

In [43]:
import os
from pathlib import Path
import json
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
OUTDIR = Path(OUTDIR)
OUTDIR.mkdir(parents=True, exist_ok=True)
print("Working dir:", Path.cwd())
print("OUTDIR:", OUTDIR)


Working dir: C:\Users\hessk\Desktop\DSC630
OUTDIR: out


In [45]:
def normalize_cols_simple(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    new_cols = []
    for c in df.columns:
        s = str(c).strip()
        s = s.replace("\n", " ").replace("\r", " ")
        s = re.sub(r"\s+", " ", s)
        s = s.lower()
        s = s.replace("%", "percent")
        s = re.sub(r"[ /()\\-]+", "_", s)
        s = re.sub(r"_+", "_", s).strip("_")
        new_cols.append(s)
    # make unique
    seen = {}
    out_cols = []
    for n in new_cols:
        if n in seen:
            seen[n] += 1
            candidate = f"{n}_{seen[n]}"
            while candidate in seen:
                seen[n] += 1
                candidate = f"{n}_{seen[n]}"
            seen[candidate] = 0
            out_cols.append(candidate)
        else:
            seen[n] = 0
            out_cols.append(n)
    df.columns = out_cols
    return df

I next put the years in chronological order, to make it easier to work with. Make sure cumulative pv witout a year still appear, only at the beginning since they are essentially a 0, and remove the columns where a year and cumulative are both missing. This is helpful because having the data organized makes it easier to work with in addition, it removes data that is mostly missing. This also follows with the standard formatting from the above step, because I do not have to worry about if Cumulative has a capital C, if the whole word is caps, they are all standard and thus I know I am capturing all the data. 

In [47]:
def safe_to_numeric(series: pd.Series, remove_commas=True, sentinels=None) -> pd.Series:
    s = series.copy()
    if remove_commas and s.dtype == object:
        s = s.str.replace(",", "", regex=False)
    out = pd.to_numeric(s, errors="coerce")
    if sentinels:
        out = out.replace(list(sentinels), np.nan)
    return out

In [49]:
def build_target_from_cumulative(df: pd.DataFrame, target_col: str = TARGET_COL, sentinels=None) -> pd.DataFrame:
    df = df.copy()
    if target_col in df.columns:
        return df

    candidates = [c for c in df.columns if re.search(r"\b20\d{2}\b", str(c)) and re.search(r"(cumul|cumulative|pv|pvs|total|detected|cum)", str(c), flags=re.I)]
    pairs = []
    for c in candidates:
        m = re.search(r"\b(20\d{2})\b", str(c))
        if m:
            pairs.append((int(m.group(1)), c))
    pairs.sort(key=lambda t: t[0])
    year_cols = [c for _, c in pairs]

    if not year_cols:
        return df

    df[year_cols] = df[year_cols].apply(lambda s: safe_to_numeric(s, remove_commas=True, sentinels=sentinels))

    cum = df[year_cols].copy()

    installs = cum.diff(axis=1)
    first_col = year_cols[0]
    installs[first_col] = cum[first_col]
    installs = installs.where(installs >= 0, np.nan)

    new_cols = []
    for c in installs.columns:
        m = re.search(r"\b(20\d{2})\b", str(c))
        year = m.group(1) if m else str(c)
        new_cols.append(f"installs_{year}")
    installs.columns = new_cols

    df = pd.concat([df, installs], axis=1)
    
    latest_col = year_cols[-1]
    df[target_col] = cum[latest_col].copy()

    return df

In [51]:
def prepare_features(df: pd.DataFrame, target_col: str = TARGET_COL):

    df = df.copy()
    id_like = [c for c in df.columns if re.search(r"(fips|id|geoid|blockgroup)", str(c), flags=re.I)]
    exclude = [c for c in df.columns if re.search(r"(cumul|cumulative|install|pv|installs_)", str(c), flags=re.I)]
    exclude = list(dict.fromkeys(exclude + [target_col] + id_like))

    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    numeric_features = [c for c in numeric_cols if c not in exclude]

    cat_features = []
    for c in df.columns:
        if c in exclude or c in numeric_features:
            continue
        try:
            nunique = df[c].nunique(dropna=True)
        except Exception:
            continue
        is_string_like = pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_categorical_dtype(df[c]) or pd.api.types.is_string_dtype(df[c])
        if is_string_like and 1 < nunique <= 50:
            cat_features.append(c)

    X = df[numeric_features + cat_features].copy()
    if target_col in df.columns:
        y = df[target_col].copy()
    else:
        y = pd.Series(dtype=float)
    return X, y, id_like, numeric_features, cat_features


In [55]:
df = normalize_cols_simple(df)
print("Columns normalized. Sample columns:")
print(list(df.columns)[:40])

Columns normalized. Sample columns:
['blockgroup_fips', 'cumulative_num_of_residential_pvs_by_2010', 'cumulative_num_of_residential_pvs_by_2011', 'cumulative_num_of_residential_pvs_by_2012', 'cumulative_num_of_residential_pvs_by_2013', 'cumulative_num_of_residential_pvs_by_2014', 'cumulative_num_of_residential_pvs_by_2015', 'cumulative_num_of_residential_pvs_by_2016', 'cumulative_num_of_residential_pvs_by_2017', 'number_of_buildings', 'avg_ghi_w_m^2', 'acs:_number_of_households_2016', 'acs:_median_household_income_inflation_adjusted_$_in_2016,_2016', 'acs:_percent_housing_units_occupied_2016', 'acs:_avg_number_of_years_of_education_2016', 'acs:_median_age', 'total_installs_detected_years']


This shows me a list of missing data. I knew this dataset was going to have a lot of missing data, just looking at it, there was information that didnt start getting collected until a little later on. Just seeing a missing count does not tell the whole story. What looks like a lot to me, might not be much for the dataset, so to fully understand the picture I wanted to compare the count to the percentage. The highest missing count, only lead to 3.4% of the data. 

In [57]:
miss = df.isna().sum().sort_values(ascending=False)
miss_percent = (miss / len(df) * 100).round(3)
missing_report = pd.DataFrame({"missing_count": miss, "missing_percent": miss_percent})
missing_report = missing_report[missing_report["missing_count"] > 0]
missing_report.to_csv(OUTDIR / "missing_report.csv")
print("Saved missing_report.csv")
numeric = df.select_dtypes(include=[np.number])
if numeric.shape[1] > 0:
    desc = numeric.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
    desc.to_csv(OUTDIR / "numeric_describe.csv")
    display(desc.head(20))
    print("Saved numeric_describe.csv")
else:
    print("No numeric columns for describe()")

Saved missing_report.csv


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
total_installs_detected_years,54493.0,5.178849,13.468532,0.0,0.0,0.0,0.0,0.0,5.0,26.0,56.0,555.0


Saved numeric_describe.csv


This checks how many rows are available for modeling to ensure there is not too many 0's, and gives another insight into how the data is shaping up after cleaing. 

In [63]:
if TARGET_COL in df.columns:
    mask = df[TARGET_COL].notna()
    df_model = df.loc[mask].reset_index(drop=True)
    print("Rows available for modeling after filtering:", df_model.shape[0], "/", len(df))
else:
    raise RuntimeError("No target column found")


Rows available for modeling after filtering: 54493 / 54493


Preparing the features

In [65]:
X, y, id_like, numeric_feats, cat_feats = prepare_features(df_model, target_col=TARGET_COL)
if TARGET_COL in df_model.columns:
    y = df_model[TARGET_COL].astype(float).reset_index(drop=True)
else:
    y = pd.Series(dtype=float)
print(f"Selected features: numeric={len(numeric_feats)} cat={len(cat_feats)} total_cols={X.shape[1]} rows={X.shape[0]}")
display(X.head(3))

Selected features: numeric=0 cat=0 total_cols=0 rows=54493


""
0
1
2


In [67]:
# Sanity check: ensure X and y align
if X.shape[0] != y.shape[0]:
    raise RuntimeError(f"X and y length mismatch: X={X.shape[0]} y={y.shape[0]}")


Using sklearn to build and preprocess the pipeline. Leveraging Onehotencoder

In [69]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="__missing__")),
    ("onehot", ohe)
])

transformers = []
if numeric_feats:
    transformers.append(("num", numeric_transformer, numeric_feats))
if cat_feats:
    transformers.append(("cat", categorical_transformer, cat_feats))
preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")

rf = RandomForestRegressor(random_state=SEED, n_jobs=-1)
pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("regressor", rf)])
print("Pipeline built.")

Pipeline built.


Splitting the dataset into 80/20

In [71]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print("Train / Test rows:", X_train.shape[0], "/", X_test.shape[0])

Train / Test rows: 43594 / 10899


Since my data was veery skewed, I had to redo some of the features. This is rebuilding a safe feature set from my Dataframe.

In [91]:
def prepare_features_fixed(df: pd.DataFrame, target_col: str = TARGET_COL, coerced_numeric_threshold=0.9):
    dfc = df.copy()
    id_like = [c for c in dfc.columns if re.search(r"(^|_)(fips|geoid|blockgroup|^id$)($|_)", str(c), flags=re.I)]
    exclude = [c for c in dfc.columns if re.search(r"(?:^|_)(?:cumul|cumulative|installs?|pv|pvs|total_installs|total_installs_detected)($|_)", str(c), flags=re.I)]
    exclude = list(dict.fromkeys(exclude + [target_col] + id_like))

    for c in dfc.columns:
        if c in exclude:
            continue
        if dfc[c].dtype == object:
            coerced = pd.to_numeric(dfc[c].str.replace(",", "", regex=False), errors="coerce")
            non_null = coerced.notna().sum()
            denom = len(coerced)
            if denom > 0 and (non_null / denom) >= coerced_numeric_threshold:
                dfc[c] = coerced
                    # now select numeric features (excluding excluded cols)
    numeric_cols = dfc.select_dtypes(include=[np.number]).columns.tolist()
    numeric_features = [c for c in numeric_cols if c not in exclude]

    cat_features = []
    for c in dfc.columns:
        if c in exclude or c in numeric_features:
            continue
        try:
            nunique = dfc[c].nunique(dropna=True)
        except Exception:
            continue
        is_string_like = pd.api.types.is_object_dtype(dfc[c]) or pd.api.types.is_categorical_dtype(dfc[c]) or pd.api.types.is_string_dtype(dfc[c])
        if is_string_like and 2 <= nunique <= 200:
            cat_features.append(c)

    X = dfc[numeric_features + cat_features].copy()
    y = dfc[target_col].copy() if target_col in dfc.columns else pd.Series(dtype=float)
    return X, y, id_like, numeric_features, cat_features

This is showing more numeric columns that I can work with. Relooking at the feature selection, numeric features, and categorical features, now that I scrapped the first chunk.

In [93]:
X_new, y_new, id_like, numeric_feats_new, cat_feats_new = prepare_features_fixed(df_model, target_col=TARGET_COL)
print("New feature selection -> numeric:", len(numeric_feats_new), " categorical:", len(cat_feats_new), " total cols:", X_new.shape[1])
print("Numeric features sample:", numeric_feats_new[:40])
print("Categorical features sample:", cat_feats_new[:40])
display(X_new.head(3))

if X_new.shape[1] == 0:
    raise RuntimeError("Still zero features selected.")

New feature selection -> numeric: 7  categorical: 0  total cols: 7
Numeric features sample: ['number_of_buildings', 'avg_ghi_w_m^2', 'acs:_number_of_households_2016', 'acs:_median_household_income_inflation_adjusted_$_in_2016,_2016', 'acs:_percent_housing_units_occupied_2016', 'acs:_avg_number_of_years_of_education_2016', 'acs:_median_age']
Categorical features sample: []


,number_of_buildings,avg_ghi_w_m^2,acs:_number_of_households_2016,"acs:_median_household_income_inflation_adjusted_$_in_2016,_2016",acs:_percent_housing_units_occupied_2016,acs:_avg_number_of_years_of_education_2016,acs:_median_age
0,587,195.559589,551,31817.877759,85.96,12.0180,35.2
1,708,195.559589,472,39054.529711,100.00,12.1300,38.7
2,905,195.559589,625,38453.439728,93.28,12.2726,31.9


In [95]:
SEED = globals().get("SEED", 42)
X = X_new.copy()
y = y_new.astype(float).reset_index(drop=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

In [99]:
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
except TypeError:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

numeric_transformer = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
categorical_transformer = Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="__missing__")), ("onehot", ohe)])
transformers = []
if numeric_feats_new:
    transformers.append(("num", numeric_transformer, numeric_feats_new))
if cat_feats_new:
    transformers.append(("cat", categorical_transformer, cat_feats_new))

preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
rf = RandomForestRegressor(random_state=SEED, n_jobs=-1)
pipeline = Pipeline([("preprocessor", preprocessor), ("regressor", rf)])

print("Fitting RandomForest on rebuilt features.")
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

Fitting RandomForest on rebuilt features.


In [105]:
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)
print(f"Results -> MAE: {mae:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")

Results -> MAE: 3.6437, RMSE: 8.6333, R2: 0.5556


C:\Users\hessk\anaconda\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


Running to check my model against the baselines. The reason I wanted to do this is because I want to make sure that even though my model is not perfect, check to see if it is better than the baseline

In [81]:
y_zero = np.zeros_like(y_test, dtype=float)
y_mean = np.full_like(y_test, fill_value=float(y_train.mean()), dtype=float)
y_median = np.full_like(y_test, fill_value=float(y_train.median()), dtype=float)

def metrics_str(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False), mean_absolute_error(y_true, y_pred)

zero_rmse, zero_mae = metrics_str(y_test, y_zero)
mean_rmse, mean_mae = metrics_str(y_test, y_mean)
med_rmse, med_mae = metrics_str(y_test, y_median)

print("\nBaselines:")
print("Zero         -> RMSE: {:.4f}, MAE: {:.4f}".format(zero_rmse, zero_mae))
print("Train-mean   -> RMSE: {:.4f}, MAE: {:.4f}".format(mean_rmse, mean_mae))
print("Train-med    -> RMSE: {:.4f}, MAE: {:.4f}".format(med_rmse, med_mae))


Model:
RandomForest -> RMSE: 8.6333, MAE: 3.6437

Baselines:
Zero         -> RMSE: 13.9109, MAE: 5.0781
Train-mean   -> RMSE: 12.9516, MAE: 6.8232
Train-med    -> RMSE: 13.9109, MAE: 5.0781


C:\Users\hessk\anaconda\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\hessk\anaconda\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\hessk\anaconda\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
C:\Users\hessk\anaconda\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the func

Since randomforest was not a great fit, it was fine, but there was room for improvement, I am trying lightGBM to see if it fits any better.
The outcome was it does not fit better and radomforest is the best overall fit. 

In [109]:
for v in ("X_train", "X_test", "y_train", "y_test"):
    if v not in globals():
        raise RuntimeError(f"{v} not found in globals. Run the prepare/split cells first.")

X_train = globals()["X_train"]
X_test = globals()["X_test"]
y_train = globals()["y_train"].astype(float).reset_index(drop=True)
y_test = globals()["y_test"].astype(float).reset_index(drop=True)
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

In [121]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

import lightgbm as lgb
model = lgb.LGBMRegressor(
    random_state=SEED,
    n_jobs=-1,
    n_estimators=2000,
    learning_rate=0.05
)

In [123]:
start = time.time()
try:
    model.fit(
        X_train, y_train_log,
        eval_set=[(X_test, y_test_log)],
        eval_metric="rmse",
        early_stopping_rounds=50,
        verbose=100
    )
    used_callbacks = False
except TypeError:
    # fallback to callbacks for older/newer wrappers
    print("LightGBM fit() did not accept early_stopping_rounds; using callbacks fallback.")
    model.fit(
        X_train, y_train_log,
        eval_set=[(X_test, y_test_log)],
        eval_metric="rmse",
        callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(period=100)]
    )
    used_callbacks = True
fit_time = time.time() - start
best_iter = getattr(model, "best_iteration_", None)
print(f"Finished fit in {fit_time:.1f}s, best_iteration_={best_iter}, used_callbacks={used_callbacks}")


LightGBM fit() did not accept early_stopping_rounds; using callbacks fallback.
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002205 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1785
[LightGBM] [Info] Number of data points in the train set: 43594, number of used features: 7
[LightGBM] [Info] Start training from score 0.899921
Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 0.735642	valid_0's l2: 0.541169
[200]	valid_0's rmse: 0.71897	valid_0's l2: 0.516917
[300]	valid_0's rmse: 0.712546	valid_0's l2: 0.507722
[400]	valid_0's rmse: 0.708718	valid_0's l2: 0.502281
[500]	valid_0's rmse: 0.706222	valid_0's l2: 0.49875
[600]	valid_0's rmse: 0.703754	valid_0's l2: 0.49527
[700]	valid_0's rmse: 0.70219	valid_0's l2: 0.493071
[800]	valid_0's rmse: 0.700904	valid_0's l2: 0.491267
[900]	valid_0's rmse: 0.700478	valid_0's l2: 0.490669
[1000]	valid_0's rmse: 0.700006	valid

In [127]:
y_pred_log = model.predict(X_test, num_iteration=best_iter if best_iter else None)
y_pred = np.expm1(y_pred_log)
y_pred = np.maximum(y_pred, 0.0)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred, squared=False)
r2 = r2_score(y_test, y_pred)
print(f"LightGBM (log1p) -> MAE: {mae:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}")


LightGBM (log1p) -> MAE: 3.2385, RMSE: 8.8846, R2: 0.5294


C:\Users\hessk\anaconda\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
